# B04 · S2 — Planificación y percepción (Célula-07)

**Objetivo (RA4-b):** planificar movimiento en el espacio de configuración (RRT) y localizar con un filtro de partículas, en Colab.

> Práctica guiada de la Sesión 2 de los [apuntes](../apuntes.md).

## 1. RRT en 2D con obstáculos

Planificación por **muestreo**: va lanzando configuraciones al azar y conectando las válidas hasta unir inicio y meta.

In [ ]:
import numpy as np

def is_free(p):
    return not (0.8 < p[0] < 1.6 and 0.8 < p[1] < 1.6)   # caja prohibida

def rrt(start, goal, bounds=((0, 4), (0, 4)), n=1500, step=0.3, seed=0):
    rng = np.random.default_rng(seed)
    nodes = {0: np.array(start, float)}; parent = {0: None}
    for i in range(1, n):
        p = np.array(goal, float) if rng.random() < 0.05 else rng.uniform(*bounds)
        k = min(nodes, key=lambda k: np.linalg.norm(nodes[k] - p))
        v = p - nodes[k]; nv = np.linalg.norm(v) + 1e-9
        new = nodes[k] + (v / nv) * min(step, nv)
        if is_free(new):
            nodes[i] = new; parent[i] = k
            if np.linalg.norm(new - np.array(goal)) < step:
                path = []; c = i
                while c is not None:
                    path.append(nodes[c]); c = parent[c]
                return path[::-1]
    return []

camino = rrt([0, 0], [3, 3])
print("puntos del camino:", len(camino))
print("inicio:", np.round(camino[0], 2), "| meta:", np.round(camino[-1], 2))

## 2. Filtro de partículas (localización)

El robot mantiene una **nube de hipótesis** (partículas) sobre dónde está y la afina con cada medida.

In [ ]:
def particle_filter(medidas, N=500, seed=1):
    rng = np.random.default_rng(seed)
    pos = rng.uniform(0, 10, N); w = np.ones(N) / N
    est = []
    for z in medidas:
        pos = pos + rng.normal(0, 0.3, N)                        # prediccion (movimiento)
        w = w * np.exp(-0.5 * ((z - pos) / 0.5) ** 2) + 1e-300    # correccion (sensor)
        w /= w.sum()
        pos = pos[rng.choice(N, N, p=w)]                          # remuestreo
        w = np.ones(N) / N
        est.append(np.average(pos, weights=w))
    return est

est = particle_filter([5.0] * 10)
print("estimaciones:", [round(e, 2) for e in est])
print("estimacion final:", round(est[-1], 3))

## 3. Simulador de robot móvil con AITK

AITK simula un robot con cámara dentro del propio notebook.

In [ ]:
%pip install aitk aitk.robots
import aitk.robots as bots

world = bots.World(220, 180, boundary_wall_color="yellow")
robot = bots.Scribbler(x=100, y=90, a=90)
robot.add_device(bots.Camera(64, 32))
world.add_robot(robot)
world.reset()
print("Mundo y robot creados. Camara:", robot["camera"])
# world.seconds(30, [controlador], real_time=True)

## Actividad guiada

1. Añade un segundo obstáculo y comprueba que el RRT lo rodea.
2. Cambia el **ruido del sensor** del filtro de partículas y observa cuántas medidas hacen falta para converger.
3. Dibuja el camino del RRT con `matplotlib` y márcalo sobre los obstáculos.

**Alcance:** espacio de configuración, métodos de planificación, plan vs. política, SLAM y filtro de partículas.